# Notebook Requirements
Run this code to execute the notebook if you didn't already cloned the repo.

In [ ]:
!git clone https://github.com/GiuseppeDaddario/Computer-Vision.git --recurse-submodules
%cd Computer-Vision

# Imports

In [ ]:
%pip install ultralytics --quiet
%pip install -U gdown

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'
from pathlib import Path
from tqdm import tqdm
import shutil
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms,models
import torchvision.transforms as T
import torchvision.datasets as datasets
from PIL import Image
from tqdm import tqdm
from torch.utils.data import random_split
import io
from google.colab import drive
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches



#from src import YOLOv5_training, YOLOv5_inference #Importing from the yolov5 repo

# Globals

In [ ]:
DATASET_PATH = "dataset/CCPD2019"
DATASET_PATH_YOLO = "dataset/CCPD2019_YOLO"

# YOLOv5 paths
TRAINING_PATH_YOLO = "dataset/ccpd_2019.yaml"
test_dir = "ccpd_challenge"
TEST_PATH_YOLOV5 = f"dataset/CCPD_YOLO/{test_dir}/images/test" 

#TODO Lore: write your paths
TRAINING_PATH_PDLPR = ""
TEST_PATH_PDLPR = ""

IMG_WIDTH = 1160
IMG_HEIGHT = 720
CLASS_ID = 0 

In [ ]:
#GLOBALS BASELINE

W_orig = 720
H_orig = 1160

W_resize = 224
H_resize =224

x_scale = W_resize/W_orig
y_scale = H_resize/H_orig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

provinces = ["皖", "沪", "津", "渝", "冀", "晋", "蒙", "辽", "吉", "黑", "苏", "浙", "京", "闽", "赣", "鲁", "豫", "鄂", "湘", "粤", "桂", "琼", "川", "贵", "云", "藏", "陕", "甘", "青", "宁", "新", "警", "学", "O"]
alphabets = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'O']
ads = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'O']

transform_detection = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
])

transform_recognition = transforms.Compose([
    transforms.Resize((64, 128)),  
    transforms.ToTensor(),
])

dataset_paths = [
    "/kaggle/input/ccpd-blur/ccpd_blur",
    "/kaggle/input/ccpd-challenge/ccpd_challenge",
    "/kaggle/input/ccpd-rotate/ccpd_rotate",
    "/kaggle/input/ccpd-db/ccpd_db",
    "/kaggle/input/ccpd-fn/ccpd_fn",
    "/kaggle/input/cppd-tilt/ccpd_tilt",
    "/kaggle/input/cppd-weather/ccpd_weather"
   
]


# Utils

## Baseline

In [ ]:
def parse_label(filename):
        parts = filename.split('-')
        str_box = parts[2]
        x1, y1 = map(int, str_box.split('_')[0].split('A'))
        x2, y2 = map(int, str_box.split('_')[1].split('A'))
        return torch.tensor([x1, y1, x2, y2], dtype=torch.float)

def compute_iou(preds, gts):
   
    intersection_x1 = np.maximum(preds[:, 0], gts[:, 0])
    intersection_y1 = np.maximum(preds[:, 1], gts[:, 1])
    intersection_x2 = np.minimum(preds[:, 2], gts[:, 2])
    intersection_y2 = np.minimum(preds[:, 3], gts[:, 3])

    intersection_w = np.maximum(0,  intersection_x2 - intersection_x1)
    intersection_h = np.maximum(0,  intersection_y2 - intersection_y1)
    intersection_area = intersection_w * intersection_h

    area_preds = (preds[:, 2] - preds[:, 0]) * (preds[:, 3] - preds[:, 1])
    area_gts = (gts[:, 2] - gts[:, 0]) * (gts[:, 3] - gts[:, 1])

    union_area = area_preds + area_gts - intersection_area

    iou = intersection_area / (union_area + 1e-7) 

    return iou

def decode_plate(indices):
    
    indices = indices.tolist() if isinstance(indices, torch.Tensor) else indices
    return (
        provinces[indices[0]] +
        alphabets[indices[1]] +
        ''.join(ads[i] for i in indices[2:])
    )

def plot_epoch_losses(epoch_losses, title="Training Loss per Epoch"):
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(epoch_losses)+1), epoch_losses, marker='o', color='blue')
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.xticks(range(1, len(epoch_losses)+1))
    plt.show()

## YOLOv5

In [ ]:
######################## UTILS FOR YOLOV5 ########################

def convert_bbox(x1, y1, x2, y2):
    """
    Converts bbox coordinates in pixels (normalized), following the YOLO format.
    """
    x_center = (x1 + x2) / 2.0 / IMG_WIDTH
    y_center = (y1 + y2) / 2.0 / IMG_HEIGHT
    width = abs(x2 - x1) / IMG_WIDTH
    height = abs(y2 - y1) / IMG_HEIGHT
    return x_center, y_center, width, height

def parse_filename(fname):
    """
    Extracts bbox coordinates from the image file name and converts them in YOLO format
    """
    parts = fname.split('-')
    if len(parts) < 4:
        return None
    bbox_part = parts[2]
    try:
        x1y1, x2y2 = bbox_part.split('_')
        x1, y1 = map(int, x1y1.split('&'))
        x2, y2 = map(int, x2y2.split('&'))
        return convert_bbox(x1, y1, x2, y2)
    except:
        return None

def process_images(images, images_src, dest_root, split):
    """
    Copies images in a new folder building the structure (splits and subfolders) required by YOLOv5.
    """
    images_dest = Path(dest_root) / "images" / split
    labels_dest = Path(dest_root) / "labels" / split
    os.makedirs(images_dest, exist_ok=True)
    os.makedirs(labels_dest, exist_ok=True)

    for img_path in tqdm(images, desc=f"Processing {split} set"):
        bbox = parse_filename(img_path.name)
        if bbox is None:
            continue

        shutil.copy(img_path, images_dest / img_path.name)
        label_path = labels_dest / (img_path.stem + ".txt")
        with open(label_path, 'w') as f:
            f.write(f"{CLASS_ID} {' '.join(f'{x:.6f}' for x in bbox)}\n")

    print(f"{split} set saved to {images_dest} and {labels_dest}")

def prepare_ccpd_base(dest_root="CCPD_YOLO", split_ratio=0.8, seed=42):
    """
    Builds the training subdataset 'ccpd_base'.
    """
    src = "ccpd_base"
    images_src = Path(f"dataset/CCPD2019/{src}")
    image_files = list(images_src.glob("*.jpg"))
    random.seed(seed)
    random.shuffle(image_files)

    split_index = int(len(image_files) * split_ratio)
    train_files = image_files[:split_index]
    val_files = image_files[split_index:]

    process_images(train_files, images_src, f"dataset/{dest_root}/{src}", "train")
    process_images(val_files, images_src, f"dataset/{dest_root}/{src}", "val")

def prepare_other_subset(subset, dest_root="CCPD_YOLO"):
    """
    Builds the other subdatasets for the testing phase (individually).
    """
    images_src = Path(f"dataset/CCPD2019/{subset}")
    image_files = list(images_src.glob("*.jpg"))
    process_images(image_files, images_src, f"dataset/{dest_root}/{subset}", "test")

##################################################################

## PDLPR

In [ ]:
######################## UTILS FOR PDLPR #########################
#TODO Lore: move this functions directly in the notebook
from src import PDLPR_training, PDLPR_inference
##################################################################

# Data

In [ ]:
SO="MacOs"
# Installing pixz for faster unxipping
if SO=="Linux":
    !apt-get update
    !apt-get install pixz
elif SO=="MacOs":
    !brew install pixz

zsh:1: command not found: apt-get


In [ ]:
# Downloading the .tar dataset and extracting it
%cd dataset
!gdown --id 1HDyFIuH65kVLtsXqxLA8gs0gJr7CRynh
!tar -I 'pixz -d' -xf CCPD2019.tar.xz

In [ ]:
#TODO Lore: move here the class for the dataset so that can be used in the every part of the code

## Baseline

In [ ]:
class baseline_dataset_detection(Dataset):
    def __init__(self, img_dir, transform=None):
        self.folder = img_dir
        self.transform = transform
        self.images = [img for img in os.listdir(img_dir) if img.endswith(".jpg")]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.folder, img_name)
        image = Image.open(img_path).convert("RGB")


        box = parse_label(img_name)
        box = np.array(box, dtype=np.float32)
        scales = np.array([x_scale / W_resize, y_scale / H_resize, x_scale / W_resize, y_scale / H_resize])
        box = box*scales
        box = torch.tensor(box,dtype=torch.float32) 

        if self.transform:
            image = self.transform(image)

        return image, box

class baseline_dataset_recognition(Dataset):
   
    def __init__(self, image_dir, transform=None):
        self.folder = image_dir
        self.images = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_name = self.images[idx]
        img_path = os.path.join(self.folder, image_name)
        
        image = Image.open(img_path).convert('RGB')

        x1,y1,x2,y2 = parse_label(image_name)
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        image = image.crop((x1, y1, x2, y2))

        if self.transform:
            image = self.transform(image)

        lp_str = image_name.split('-')[-3]
        lp_idx = list(map(int, lp_str.split('_')))

    
        return image, torch.tensor(lp_idx)

## YOLOv5

In [ ]:
base_dest = "CCPD_YOLO"

other_subsets = [
    "ccpd_blur", "ccpd_challenge", "ccpd_db",
    "ccpd_fn", "ccpd_np", "ccpd_rotate", "ccpd_tilt", "ccpd_weather"
]

# Build the training subset (both train-val splits)
prepare_ccpd_base(dest_root=base_dest)

# Other subsets (only for testing)
for subset in other_subsets:
    prepare_other_subset(subset, dest_root=base_dest)

## PDLRP

In [ ]:
#TODO Lore: move here the actual preprocessing needed (the function calls)

# Network

## Baseline

In [ ]:
class detection_model(nn.Module):
    
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = models.resnet34(pretrained=pretrained)                

        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  

        self.pool = nn.AdaptiveAvgPool2d((1, 1))  

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 4),
            nn.Sigmoid()  
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = self.regressor(x)
        return x

class recognition_model(nn.Module):
    def __init__(self, num_classes_list):
        super(recognition_model, self).__init__()
        base_model = models.resnet34(pretrained=True)
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])  
        self.classifiers = nn.ModuleList([
            nn.Linear(512, n_classes) for n_classes in num_classes_list
        ])

    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.view(x.size(0), -1)
        outputs = [clf(x) for clf in self.classifiers]
        return outputs

## YOLOv5

## PDLRP

In [ ]:
#TODO Lore: move here the model architecture

# Train

## Baseline

In [ ]:
#TRAINING DETECTION MODEL

number_samples =100000
num_epochs = 10


train_folder = "/kaggle/input/cppdbase/temp_rinominati"

train_dataset_detection = baseline_dataset_detection(train_folder, transform_detection)
train_dataset_detection,_ = random_split(train_dataset_detection, [number_samples,len(train_dataset_detection)-number_samples ])

train_loader_detection = DataLoader(
    train_dataset_detection,
    batch_size=32,
    shuffle=True,
    num_workers=4,  
    pin_memory=True
)

model_detection = detection_model().to(device)
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model_detection.parameters(), lr=1e-3)
epoch_losses = []

for epoch in range(num_epochs):
    model_detection.train()
    epoch_loss =0
    num_batches=0
    
    for imgs,targets in tqdm(train_loader_detection, desc=f"Epoch {epoch+1}/{num_epochs}"):
        imgs, targets = imgs.to(device), targets.to(device)
        preds = model_detection(imgs)
        loss = criterion(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1


    avg_loss = epoch_loss / num_batches
    epoch_losses.append(avg_loss)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.12f}")
plot_epoch_losses(epoch_losses)

In [ ]:
#IF YOU WANT TO SAVE THE DETECTION MODEL

torch.save(model_detection.state_dict(), "/content/drive/MyDrive/resNet34_30epoch_16batch.pth")

In [ ]:
def train_recognition(model, dataloader, criterion, optimizer, epochs=30):
    model.train()
    epoch_losses = []
    
    for epoch in range(epochs):
        
        epoch_loss=0
        num_batches = 0
        for imgs, labels in tqdm(dataloader):
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = sum(criterion(out, labels[:, i]) for i, out in enumerate(outputs))
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches
        epoch_losses.append(avg_loss)

        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.12f}")
    plot_epoch_losses(epoch_losses)  

In [ ]:
#TRAINING RECOGNITION MODEL


train_folder = "/kaggle/input/cppdbase/temp_rinominati"

train_dataset_recognition = baseline_dataset_recognition(train_folder, transform=transform_recognition)
train_loader_recognition = DataLoader(train_dataset_recognition, batch_size=32, shuffle=True)

num_classes = [len(provinces), len(alphabets)] + [len(ads)] * 5

model_recognition = recognition_model(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_recognition.parameters(), lr=1e-4)

train_recognition(model_recognition, train_loader_recognition, criterion, optimizer, epochs=10)

In [ ]:
#IF YOU WANT TO SAVE THE RECOGNITION MODEL

torch.save(model_recognition.state_dict(), "/kaggle/working/model_detection.pth")

## YOLOv5

In [ ]:
# Train YOLOv5s
YOLOv5_training(
    weights="yolov5s.pt",
    data=DATASET_PATH_YOLO,
    epochs=300,
    batch_size=50,
    imgsz=640,
    optimizer="Adam",
    lr0=1e-3,
    lrf=1e-5,
    cos_lr=True,
    project="runs/train",
    name="lp_detection",
    cache="ram"
)

## PDLRP

In [ ]:
# ------ training ------ #
#TODO Lore: check this works
print("PDLPR Training ...")
PDLPR_training(TRAINING_PATH_PDLPR, batch_size=32, num_epochs=3)

# Evaluation

## Baseline

In [ ]:
def test_detection(model, dataset, batch_size):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    total_iou = 0.0
    well_predicted = 0
    total_samples = 0

    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Testing"):
            imgs = imgs.cuda()
            preds = model(imgs).cpu().numpy()

            target_bboxes = targets.numpy()

            ious = compute_iou(preds, target_bboxes)
            total_iou += ious.sum()
            well_predicted += (ious > 0.7).sum()    
            total_samples += len(imgs)

    avg_iou = total_iou / total_samples
    print(f"\nAverage value of IoU on {total_samples} samples: {avg_iou:.4f}")
    print(f"Accuracy of well-predicted samples (IoU > 0.7): {100 * well_predicted / total_samples:.2f}%")
    print(f"Total number of samples tested: {total_samples}")

In [ ]:
# If you want to test a pre-trained model

model_detection = detection_model()  
model_detection.load_state_dict(torch.load("/kaggle/input/detection_model/pytorch/default/1/resNet34_30epoch_16batch (1).pt"))
model_detection = model_detection.to(device)

In [ ]:
for path in dataset_paths:

    print(f"\n Test on {path}")
    test_dataset_detection = baseline_dataset_detection(path, transform=transform_detection)
    test_detection(model_detection, test_dataset_detection, batch_size=16)

In [ ]:
def test_recognition(model, dataloader):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for imgs, labels in tqdm(dataloader):
            imgs = imgs.to(device)
            labels = labels.to(device)  

            outputs = model(imgs)  

            preds = [torch.argmax(output, dim=1) for output in outputs]  
            preds = torch.stack(preds, dim=1)  
            
            matches = (preds == labels).all(dim=1)  

            correct += matches.sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"\nTotal accuracy: {acc * 100:.2f}%")
    print(f"Total number of samples tested: {total}")

In [ ]:
#IF YOU WANT TO USE A PRE-TRAINED MODEL

num_classes = [len(provinces), len(alphabets)] + [len(ads)] * 5
model_recognition = recognition_model(num_classes)
model_recognition.load_state_dict(torch.load("/kaggle/input/recognition_model/pytorch/default/1/model_weights (1).pth"))  
model_recognition.to(device)

In [ ]:
for path in dataset_paths:
  print(f"\n Test on {path}")
  test_dataset_recognition = baseline_dataset_recognition(path, transform=transform_recognition)
  test_loader_recognition = DataLoader(test_dataset_recognition, batch_size=32, shuffle=False)
  test_recognition(model_recognition, test_loader_recognition)

In [ ]:
def complete_pipeline_test(model_detection, model_recognition, dataset_path, iou_threshold=0.6, batch_size=32):
    
    model_detection.eval()
    model_recognition.eval()

    model_detection.to(device)
    model_recognition.to(device)


    dataset = baseline_dataset_detection(dataset_path, transform=transform_detection)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


    total_images = 0
    correct_predictions = 0
    skipped_images = 0

    with torch.no_grad():
        for batch_idx, (batch_imgs, batch_true_bboxes) in enumerate(tqdm(dataloader, desc="Testing total pipeline")):
            batch_imgs = batch_imgs.to(device)
            batch_preds = model_detection(batch_imgs).cpu().numpy()
            batch_true_bboxes = batch_true_bboxes.numpy()
            batch_size_actual = batch_imgs.size(0)

             
            batch_preds[:, [0, 2]] *= W_resize
            batch_preds[:, [1, 3]] *= H_resize
            abs_true = batch_true_bboxes * np.array([W_resize, H_resize, W_resize, H_resize])

            ious = compute_iou(batch_preds, abs_true)

            for i in range(batch_size_actual):
                total_images += 1

                if ious[i] < iou_threshold:
                    skipped_images += 1
                    continue

                global_idx = batch_idx * batch_size + i
                img_path = os.path.join(dataset_path, dataset.images[global_idx])
                original_img = Image.open(img_path).convert("RGB")

                pred_bbox = batch_preds[i]
                x1, y1, x2, y2 = map(int, pred_bbox)
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(W_resize, x2), min(H_resize, y2)
                
                x1 = x1/x_scale
                y1 = y1/y_scale
                x2 = x2/x_scale
                y2 = y2/y_scale
                cropped_img = original_img.crop((x1, y1, x2, y2))
                cropped_img = transform_recognition(cropped_img).unsqueeze(0).to(device)

                outputs = model_recognition(cropped_img)
                pred = torch.stack([torch.argmax(out, dim=1) for out in outputs], dim=1).squeeze(0)

            
                filename = dataset.images[global_idx]
                label_str = filename.split('-')[-3]
                true_label = torch.tensor(list(map(int, label_str.split('_')))).to(device)

                if torch.equal(pred, true_label):
                    correct_predictions += 1

    valid_images = total_images - skipped_images
    accuracy = correct_predictions / total_images

   
    print(f"Total number of samples: {total_images}")
    print(f"Bounding box valid (IoU > {iou_threshold}): {valid_images}")
    print(f"Total number of well-predicted license plates: {correct_predictions}")
    print(f"Final accuracy: {accuracy * 100:.2f}%")

In [ ]:
#IF YOU WANT TO USE PRE-TRAINED MODELS

model_detection = detection_model()
model_detection.load_state_dict(torch.load("/kaggle/input/detection_model/pytorch/default/1/resNet34_30epoch_16batch (1).pth"))


num_classes = [len(provinces), len(alphabets)] + [len(ads)] * 5
model_recognition = recognition_model(num_classes)
model_recognition.load_state_dict(torch.load("/kaggle/input/recognition_model/pytorch/default/1/model_weights (1).pth"))  


In [ ]:
for path in dataset_paths:
   print(f"\n Test on {path}")
   complete_pipeline_test(model_detection, model_recognition, path)

## YOLOv5

In [ ]:
YOLOv5_inference(
    weights="yolov5/runs/train/exp/weights/best.pt",
    source=TEST_PATH_YOLOV5,  # cartella con almeno 5 immagini
    imgsz=640,
    device="cuda:0",  # o "cpu"
    project="runs/detect",
    name="lp_test",
    exist_ok=True
)

## PDLRP

In [ ]:
# ------ inference ------ #
#TODO Lore: check this works
print("PDLPR Inference ...")
PDLPR_inference(TEST_PATH_PDLPR, batch_size=64)